In [1]:
import pandas as pd
import os
os.chdir("../")
from src.customer_churn.logging.logger import logging

## 1. Import the libraries:

In [2]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# 2. Import the training data:

In [3]:
train = pd.read_parquet("Artifacts/data_cleaning/data_clipped/train.parquet")

In [4]:
train.head(2)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,order_issue,quantity_abs,quantity_abs_clipped,price_clipped
0,532371,21212,PACK OF 72 RETROSPOT CAKE CASES,48,2010-11-11 17:55:00,0.55,15358.0,United Kingdom,0,48,48,0.55
1,532810,22560,TRADITIONAL MODELLING CLAY,1,2010-11-14 15:23:00,1.25,17545.0,United Kingdom,0,1,1,1.25


# 3. Define the churn label:

- The transactions after the date `2011-03-31` are already dropped during data ingestion.
- The **observation window** would be `2009-12-01`(this is the first purchase invoice date recorded in the dataset) till `2010-12-31`
- The **churn window** would be **3 months + the observation window** i.e. `2011-01-01` to `2011-03-31`

#### Logic implemented for defining the churn label:
- The churn window will be used as a filter to define the churn(1) or retained(0)
- If customer didn't make a single purchase in **the churn window** then 1(churn) else 0(retained)

#### Steps involved:
1. The `dataframe` will be split into two dataframes: `df with observation window(observation_df)` and `df with churn window(churn_df)`
2. Get the unique `customerid` from `churn_df`
3. Create a new feature `churn` on `observation_df`, if `customerid` is in `customerid` of `churn_df` then 0 (retained) else 1(churn).

## 3.1 Add the churn feature:

In [11]:
churn_cutoff_date = '2011-03-31'
chun_cutoff_date = pd.to_datetime(churn_cutoff_date)
observation_date = chun_cutoff_date - pd.DateOffset(months=3)

In [12]:
print(f"The observation window will be until: {observation_date}")

The observation window will be until: 2010-12-31 00:00:00


In [13]:
print(f"The churn window will be until: {chun_cutoff_date}")

The churn window will be until: 2011-03-31 00:00:00


In [14]:
observation_df = train[train['invoicedate'] <= observation_date].copy()
churn_df = train[(train['invoicedate'] > observation_date) & (train['invoicedate'] <= chun_cutoff_date)].copy()

In [15]:
observation_df.shape

(341359, 12)

In [16]:
churn_df.shape

(54663, 12)

In [17]:
print(f"The min invoice date in observation df: {observation_df['invoicedate'].min()} and the max is: {observation_df['invoicedate'].max()}")
print(f"{'-' * 25}")

customer_ids_from_observation_df = set(observation_df['customerid'].unique())
print(f"There are a total of: {len(customer_ids_from_observation_df)} customer IDs who made the transaction in observation window.")

The min invoice date in observation df: 2009-12-01 07:45:00 and the max is: 2010-12-23 16:06:00
-------------------------
There are a total of: 4382 customer IDs who made the transaction in observation window.


In [18]:
print(f"The min invoice date in churn df: {churn_df['invoicedate'].min()} and the max is: {churn_df['invoicedate'].max()}")

The min invoice date in churn df: 2011-01-04 10:00:00 and the max is: 2011-03-30 17:33:00


In [19]:
retained_customer_ids = set(churn_df['customerid'].unique())
print(f"There are a total of: {len(retained_customer_ids)} customer IDs who made the transaction in churn window.")

There are a total of: 1781 customer IDs who made the transaction in churn window.


In [20]:
observation_df['churn'] = (~observation_df['customerid'].isin(retained_customer_ids)).astype(int)

In [22]:
observation_df.head(2)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,order_issue,quantity_abs,quantity_abs_clipped,price_clipped,churn
0,532371,21212,PACK OF 72 RETROSPOT CAKE CASES,48,2010-11-11 17:55:00,0.55,15358.0,United Kingdom,0,48,48,0.55,0
1,532810,22560,TRADITIONAL MODELLING CLAY,1,2010-11-14 15:23:00,1.25,17545.0,United Kingdom,0,1,1,1.25,1


In [23]:
observation_df['churn'].value_counts()

churn
0    205243
1    136116
Name: count, dtype: int64

## 3.2 Transform into function:

In [24]:
def define_churn_label(df:pd.DataFrame, churn_cutoff_date:str, data_name:str) -> pd.DataFrame:
    """
    Adds a new feature churn label: 1(Churn) & 0(Retain)
    Parameters:
    -----------
    df: DataFrame at customer level aggregation where churn feature needs to be added
    churn_cutoff_date: The churn window end date
    """
    # Set the observation and churn date:
    chun_cutoff_date = pd.to_datetime(churn_cutoff_date)
    observation_date = chun_cutoff_date - pd.DateOffset(months=3)
    logging.info(f"Churn window frame from {observation_date + pd.DateOffset(days=1)} : {churn_cutoff_date}")
    logging.info(f"Observation window is set till: {observation_date}")    

    # Split the data into observation and churn window dataframe:
    observation_df = df[df['invoicedate'] <= observation_date].copy()
    churn_df = train[(df['invoicedate'] > observation_date) & (df['invoicedate'] <= chun_cutoff_date)].copy()
    customer_ids_from_observation_df = set(observation_df['customerid'].unique())

    # Logs:
    logging.info(f"Observation df of shape: ({observation_df.shape}) and churn df of shape: ({churn_df.shape}) created.")
    logging.info(f"Observation df has invoice date of min : {observation_df['invoicedate'].min()} and max:       {observation_df['invoicedate'].max()} | Churn df has invoice date of min : {churn_df['invoicedate'].min()} and max:{churn_df['invoicedate'].max()}")

    # Labelling churn:
    observation_df['churn'] = (~observation_df['customerid'].isin(customer_ids_from_churn_df)).astype(int)
    
    # Logs:
    logging.info(f"churn feature created | The count of churn(1) and retain(0) is:\n{observation_df['churn'].value_counts()}")
    
    return df
    
    

In [25]:
observation_df.head(2)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,order_issue,quantity_abs,quantity_abs_clipped,price_clipped,churn
0,532371,21212,PACK OF 72 RETROSPOT CAKE CASES,48,2010-11-11 17:55:00,0.55,15358.0,United Kingdom,0,48,48,0.55,0
1,532810,22560,TRADITIONAL MODELLING CLAY,1,2010-11-14 15:23:00,1.25,17545.0,United Kingdom,0,1,1,1.25,1


# 4. Start Creating New Features:

## 1. Product and customer level metrics:

### 1.1 Add `Total number of orders, total number of order issues, and total order issue rate` at `stockcode` level:

Here, stock level metrics will be calculated for each unique stockcode, those are:
1. total_n_orders
2. total_n_order_issues
3. total_order_issue_rate

In [30]:
stock_code_df = observation_df[['invoice', 'stockcode', 'quantity', 'order_issue']].copy()

In [34]:
print(f"There are a total of {stock_code_df['stockcode'].nunique():,} unique products.")

There are a total of 4,009 unique products.


Adding `total_n_orders` and `total_n_order_issues per stockcode`:

In [32]:
stock_code_df_aggregated = stock_code_df.groupby('stockcode').agg(
    total_n_orders = ('invoice', 'nunique'),
    total_n_order_issues = ('order_issue', 'sum')).reset_index()

In [33]:
stock_code_df_aggregated.head(2)

,stockcode,total_n_orders,total_n_order_issues
0,10002,221,6
1,10080,5,0


Adding `total_order_issue_rate`:

In [37]:
stock_code_df_aggregated['total_order_issue_rate'] = stock_code_df_aggregated['total_n_order_issues'] / stock_code_df_aggregated['total_n_orders']

In [38]:
stock_code_df_aggregated.head(2)

,stockcode,total_n_orders,total_n_order_issues,total_order_issue_rate
0,10002,221,6,0.027149
1,10080,5,0,0.000000


### 1.2 Product level metrics for each product bought by customer:

Here, first a new df will be created which will hold each customerid and the stockcode then will be joined with 
`stock_code_df` to pull the stockcode/product level metrics that was calculated earlier:

`product_per_customer` table holds the information about the `customerid` and all the `stockcode` of the products they have purchased:

In [39]:
product_per_customer = observation_df[['customerid', 'stockcode']].drop_duplicates()

In [40]:
product_per_customer.shape

(238842, 2)

In [43]:
product_per_customer.head(2)

,customerid,stockcode
0,15358.0,21212
1,17545.0,22560


The `customer_and_prod_lvl_metrics` and `stock_code_df_aggregated` dataframe will be joined on `stockcode` with 'right' join
to pull the product level metrics for all the products bought by customers which are identified by their `stockcode` 

In [44]:
customer_and_prod_lvl_metrics = stock_code_df_aggregated.merge(right=product_per_customer,
                                                               how='right',
                                                              on='stockcode')

In [86]:
customer_and_prod_lvl_metrics.head(2)

,stockcode,total_n_orders,total_n_order_issues,total_order_issue_rate,customerid
0,21212,1280,13,0.010156,15358.0
1,22560,276,2,0.007246,17545.0


In [96]:
customer_risk_at_product_lvl = customer_and_prod_lvl_metrics.groupby(['customerid']).agg(
    avg_stockcode_issue_rate=('total_order_issue_rate', 'mean'),
    max_stockcode_issue_rate=('total_order_issue_rate', 'max'),
    total_orders_made_for_stock = ('total_n_orders', 'sum')
).reset_index()

# Now work on these two DFs below:

In [97]:
customer_risk_at_product_lvl.head(2)

,customerid,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
0,12346.0,0.098834,1.053333,6700
1,12347.0,0.024199,0.102564,11738


In [89]:
observation_df.head(2)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,order_issue,quantity_abs,quantity_abs_clipped,price_clipped,churn
0,532371,21212,PACK OF 72 RETROSPOT CAKE CASES,48,2010-11-11 17:55:00,0.55,15358.0,United Kingdom,0,48,48,0.55,0
1,532810,22560,TRADITIONAL MODELLING CLAY,1,2010-11-14 15:23:00,1.25,17545.0,United Kingdom,0,1,1,1.25,1


In [93]:
observation_df.columns

Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customerid', 'country', 'order_issue', 'quantity_abs',
       'quantity_abs_clipped', 'price_clipped', 'churn'],
      dtype='object')